# Pretrained Models, Fine-Tuning, Prompting, and RAG

> Understand common LLM workflows without requiring an API key or downloading a model.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Distinguish prompting, fine-tuning, and retrieval-augmented generation (RAG).
- Understand the role of tokenization and task-specific evaluation.
- Implement a tiny local retrieval example using scikit-learn TF-IDF.

This notebook does not call a paid API or download a large checkpoint. Optional Hugging Face code is provided as a reference; compute, access terms, and model licenses should be checked before using a real model.

## Three different approaches
- **Prompting:** change instructions and context provided at inference time; model weights remain fixed.
- **Fine-tuning:** continue training a pretrained model on task/domain examples; requires curated data and evaluation.
- **RAG:** retrieve relevant external evidence and pass it to a generator. RAG can improve grounding but does not guarantee correctness; retrieval and answer quality must both be evaluated.

Hugging Face's current Transformers guide provides a `Trainer` workflow for training/fine-tuning: https://huggingface.co/docs/transformers/trainer

In [1]:
# A tiny local retrieval baseline: retrieve the document most similar to a query.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "A train/test split estimates generalization on data not used to fit the model.",
    "A scikit-learn Pipeline keeps preprocessing and the estimator together.",
    "Precision measures the fraction of predicted positives that are truly positive.",
    "Recall measures the fraction of true positives that the model identifies.",
]
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(documents)
query = "How do I keep feature scaling from leaking test information?"
query_vec = vectorizer.transform([query])
scores = cosine_similarity(query_vec, doc_matrix).ravel()
for idx in scores.argsort()[::-1][:2]:
    print(f"score={scores[idx]:.3f} | {documents[idx]}")

score=0.341 | A train/test split estimates generalization on data not used to fit the model.
score=0.000 | Recall measures the fraction of true positives that the model identifies.


## Evaluate retrieval separately from generation
For a retrieval system, create question-answer pairs with source documents. Measure whether relevant sources appear in top-k (Recall@k / MRR), and separately judge whether the final answer is supported by the retrieved evidence. Keep a “no answer found” path for questions not covered by the corpus.

In [2]:
# Minimal evidence-aware answer assembly (not an LLM call).
best_idx = int(scores.argmax())
retrieved_text = documents[best_idx]
answer_template = f"Based on the retrieved note: {retrieved_text}"
print(answer_template)
print("Retrieved document index:", best_idx)

Based on the retrieved note: A train/test split estimates generalization on data not used to fit the model.
Retrieved document index: 0


## Optional Hugging Face reference (not executed)
Install separately and choose a small, appropriately licensed checkpoint. Model loading may require network access and several gigabytes of disk or memory.

```python
# Optional: install `transformers` and `torch` in a separate environment first.
# from transformers import pipeline
# classifier = pipeline("text-classification", model="YOUR_APPROVED_MODEL_ID")
# print(classifier("The documentation was clear and helpful."))
```

Never paste secrets into notebooks. Keep API keys in environment variables or a secret manager; do not commit them to GitHub.

## Practice
1. Add a document that directly addresses the query and observe ranking changes.
2. Describe one case where RAG may still generate a wrong answer.
3. Compare what changes in prompting versus fine-tuning.